# 01 · EDA — Battery Degradation

Exploratory analysis of the synthetic lithium-ion dataset: capacity fade,
internal-resistance growth, and thermal exposure across usage profiles.

> All data is synthetic. No Apple confidential data is used.


In [ ]:
# Make the project importable when running the notebook from notebooks/
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import config
pd.set_option('display.width', 120)

In [ ]:
# Build the processed data if the pipeline hasn't been run yet.
from src.ingest.load_raw_data import load
from src.features.build_features import build as build_features
if not config.CYCLE_FEATURES_CSV.exists():
    load(); build_features()
cycles = pd.read_csv(config.CYCLES_CSV)
factory = pd.read_csv(config.FACTORY_CSV)
usage = pd.read_csv(config.USAGE_CSV)
failures = pd.read_csv(config.FAILURES_CSV)
print('cycles', cycles.shape, '| cells', factory.shape[0])

## Capacity fade curves (sample cells)


In [ ]:
init = cycles.sort_values('cycle_index').groupby('cell_id').head(5).groupby('cell_id')['discharge_capacity_ah'].mean()
cyc = cycles.merge(init.rename('init_cap'), on='cell_id')
cyc['soh'] = cyc['discharge_capacity_ah'] / cyc['init_cap']
prof = usage.set_index('cell_id')['usage_profile']
fig, ax = plt.subplots(figsize=(9,5))
for cid in cycles['cell_id'].drop_duplicates().sample(12, random_state=1):
    g = cyc[cyc.cell_id==cid]
    ax.plot(g.cycle_index, g.soh, lw=1, alpha=.8, label=f'{cid} ({prof.get(cid,"")})')
ax.axhline(0.8, ls='--', color='red', label='EOL (80% SOH)')
ax.set_xlabel('cycle'); ax.set_ylabel('SOH'); ax.set_title('Capacity fade trajectories')
ax.legend(fontsize=7, ncol=2); plt.show()

## SOH distribution by usage profile


In [ ]:
last = cyc.sort_values('cycle_index').groupby('cell_id').tail(1)
last = last.merge(usage[['cell_id','usage_profile']], on='cell_id')
last.boxplot(column='soh', by='usage_profile', figsize=(9,5), rot=20)
plt.suptitle(''); plt.title('Final SOH by usage profile'); plt.ylabel('SOH'); plt.show()

## Internal resistance growth vs cycle


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
for cid in cycles['cell_id'].drop_duplicates().sample(10, random_state=3):
    g = cycles[cycles.cell_id==cid]
    ax.plot(g.cycle_index, g.internal_resistance_mohm, lw=1, alpha=.7)
ax.set_xlabel('cycle'); ax.set_ylabel('internal resistance (mOhm)')
ax.set_title('Impedance growth over life'); plt.show()

## Failure / escalation overview


In [ ]:
print(failures['failure_severity'].value_counts())
print('\nEscalation rate:', round(failures['escalation_required'].mean(), 3))
failures[['capacity_drop_event','impedance_spike_event','thermal_anomaly_event','early_degradation_flag']].mean().plot.bar(figsize=(7,4), title='Event flag rates'); plt.show()

**Takeaways:** capacity fade accelerates after a knee; thermal-stress and
fast-charge profiles fade fastest and grow resistance quickest; these signals
drive the SOH/RUL/failure models in the next notebooks.
